[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/autoscientist/autoscientist.ipynb)

# AutoScientist: Automating the Science of Model Training

Upload a Hugging Face dataset, train with AutoScientist (with augmentation), and download the best checkpoint.

**Docs:** [/autoscientist-quickstart/](https://docs.adaptionlabs.ai/autoscientist-quickstart/) · [tutorial](https://docs.adaptionlabs.ai/tutorials/train-with-autoscientist/)  
**Blog:** [AutoScientist: Automating the Science of Model Training](https://adaptionlabs.ai/blog/autoscientist) · [AutoScientist API: Automated Model Training](https://adaptionlabs.ai/blog/autoscientist-api)  
**Prerequisites:** an Adaption API key ([create one](https://adaptionlabs.ai/app/settings?tab=api_keys)) set as `ADAPTION_API_KEY` (env var or Colab secret).

> ⏱ Runtime: hours · 💳 Cost: consumes training credits

In [ ]:
%pip install -q adaption

In [ ]:
import os

from adaption import Adaption

if "ADAPTION_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # Colab secrets

        os.environ["ADAPTION_API_KEY"] = userdata.get("ADAPTION_API_KEY")
    except ImportError:
        from getpass import getpass

        os.environ["ADAPTION_API_KEY"] = getpass("ADAPTION_API_KEY: ")

client = Adaption()

## Prepare an adapted dataset

---
Here we will upload the [`alpaca`](https://huggingface.co/datasets/tatsu-lab/alpaca) instruction dataset from HuggingFace.


In [ ]:
upload_response = client.datasets.create_from_huggingface(
    url="https://huggingface.co/datasets/tatsu-lab/alpaca",
    files=["data/train-00000-of-00001-a09b74b3ef9c3b56.parquet"],
)
print(upload_response.dataset_id)

Wait for ingestion to finish (row_count populated)

In [ ]:
import time

while True:
    status = client.datasets.get_status(upload_response.dataset_id)
    if status.status == "failed":
        err = status.error_data
        msg = (err and err.message) or "unknown error"
        raise RuntimeError(f"Ingestion failed: {msg}")
    if status.row_count is not None:
        break
    time.sleep(5)

## Train with AutoScientist

The platform selects the model, recipe, iteration limit, target win rate, and training columns. See the [AutoScientist guide](https://docs.adaptionlabs.ai/guides/autoscientist/) for a more detailed explanation.

### Data augmentation

> Augmentation is optional, but strongly recommended when your dataset is small or lacks diversity. Additional training rows give the model more examples to learn from, helping reduce overfitting and the loss of general capabilities during training.


We recommend targeting at least 20,000 domain rows and maintaining 40% general purpose datapoints to preserve model capabilities:

- **Domain rows** add additional data specific to your dataset's domain, strengthening the behavior you want the model to learn. Your existing dataset counts toward the 20,000-row target.
- **General rows** add diverse, general-purpose examples that help the model retain broader capabilities.

The rows are added to the training dataset before AutoScientist starts its training iterations. Use `augmentation_domain_rows` and `augmentation_general_rows` to request them:

In [ ]:
dataset = client.datasets.get(dataset_id=upload_response.dataset_id)

In [ ]:
domain_rows_to_add = max(0, 20_000 - dataset.row_count)
total_domain_rows = dataset.row_count + domain_rows_to_add

In [ ]:
import math

target_general_ratio = 0.4
general_rows_to_add = math.ceil((target_general_ratio * total_domain_rows) / (1 - target_general_ratio))
print(total_domain_rows, (domain_rows_to_add, general_rows_to_add))

In [ ]:
run = client.autoscientist.create(
    dataset_id=dataset.dataset_id,
    augmentation_domain_rows=domain_rows_to_add,
    augmentation_general_rows=general_rows_to_add,
)
print(run.id, run.status)

### Wait for completion

In [ ]:
run = client.autoscientist.wait_for_completion(run.id)
print(run.status, run.best_win_rate)

if run.status != "succeeded":
    raise RuntimeError(f"AutoScientist ended with {run.status}")

### Download model

In [ ]:
if not run.download_available:
    raise RuntimeError("The checkpoint is not available for download")

with client.autoscientist.with_streaming_response.download(run.id) as response:
    response.stream_to_file("best-checkpoint.tgz")